# 🧪 Lesson 15 — Rule-Based Drug-Likeness and Molecular Properties

> **Author:** Omar Hamouda  
> **Environment:** WSL/Ubuntu | VS Code | Conda (biochem)  
> **Purpose:** Evaluate molecular properties and rule-based drug-likeness of the current standardized docking Top 5.

---

## 📌 Overview

The standardized docking workflow ranks 111 validated ligands against Trypsin (2ZQ2).

This lesson evaluates the **current Top 5** using RDKit molecular descriptors and three rule-based filters:

- **Lipinski Rule of Five**
- **Veber rules**
- **PAINS filtering**

These analyses assess molecular properties and rule-based drug-likeness. They are **not full ADME or toxicity predictions**.

---

## 🎯 Current Standardized Top 5

The current Top 5 are derived dynamically from:

`docking/results/standardized_affinities.csv`

The current standardized ranking is:

| Rank | Ligand | Affinity (kcal/mol) |
|------|--------|---------------------|
| 1 | R11 | -9.834149 |
| 2 | 13U | -9.438036 |
| 3 | BAH | -9.420844 |
| 4 | 12U | -9.339968 |
| 5 | 607 | -9.332170 |

These docking scores are computational ranking scores and should not be interpreted as experimental binding affinities.

---

## 📁 Required Files

- `docking/results/standardized_affinities.csv`
- `docking/results/fixed/<ligand>_fixed.sd` for each current Top 5 ligand

The authoritative implementation of this analysis is:

`scripts/admet_top5.py`

---

## 🧬 Lipinski's Rule of 5

A molecule is evaluated against the following rule-based thresholds:

| Property | Rule |
|----------|------|
| **Molecular Weight** | ≤ 500 Da |
| **LogP** | ≤ 5 |
| **H-Bond Donors** | ≤ 5 |
| **H-Bond Acceptors** | ≤ 10 |

**Rule-based verdict:**

- **0 violations** → Excellent
- **1 violation** → Acceptable
- **2+ violations** → Poor

---

## 🧬 Veber Rules

The project evaluates:

| Property | Rule |
|----------|------|
| **Rotatable Bonds** | ≤ 10 |
| **TPSA** | ≤ 140 Å² |

---

**Let's start!** 🚀


In [1]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())

Working directory set to: /home/omar/biochemist-python_ORGANIZED


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 2: Setup & Imports
# ═══════════════════════════════════════════════════════════

import os
import warnings
warnings.filterwarnings("ignore")

# Data manipulation
import pandas as pd
import numpy as np

# Cheminformatics
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors

# PAINS filter
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

# Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib.cm as cm

# ─── Report versions ───
print("=" * 60)
print("  Environment Check")
print("=" * 60)
print(f"\n✅ RDKit:      {Chem.rdBase.rdkitVersion}")
print(f"✅ pandas:     {pd.__version__}")
print(f"✅ numpy:      {np.__version__}")
print("\n" + "=" * 60)
print("  🎉 All tools ready!")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Load Current Standardized Top 5 Ligands
# ═══════════════════════════════════════════════════════════

DOCKING_RESULTS = "docking/results/standardized_affinities.csv"
FIXED_DIR = "docking/results/fixed"
TOP_N = 5

ranking = pd.read_csv(DOCKING_RESULTS)

ranking["affinity"] = pd.to_numeric(
    ranking["affinity"],
    errors="coerce",
)

ranking = ranking[
    ranking["status"].isin(["ok", "existing"])
].dropna(subset=["affinity"])

ranking = (
    ranking
    .sort_values("affinity")
    .head(TOP_N)
    .reset_index(drop=True)
)

TOP_HITS = ranking["ligand_id"].tolist()

ligands = {}

print("=" * 70)
print("  Current Standardized Top 5 Ligands")
print("=" * 70)

print(
    ranking[
        ["ligand_id", "affinity", "status"]
    ].to_string(index=False)
)

for ligand_id in TOP_HITS:
    path = os.path.join(
        FIXED_DIR,
        f"{ligand_id}_fixed.sd",
    )

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Prepared ligand file not found: {path}"
        )

    mol = Chem.MolFromMolFile(
        path,
        removeHs=False,
        sanitize=True,
    )

    if mol is None:
        raise ValueError(
            f"RDKit could not parse: {path}"
        )

    ligands[ligand_id] = mol

    print(
        f"✅ {ligand_id:5s} — "
        f"{mol.GetNumHeavyAtoms()} heavy atoms"
    )

print(
    f"Loaded: {len(ligands)} / {TOP_N} ligands"
)

assert len(ligands) == TOP_N


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 4: Compute Molecular Descriptors
# ═══════════════════════════════════════════════════════════

def compute_descriptors(mol):
    """Compute all ADMET-relevant descriptors for a molecule."""
    return {
        "MW":             round(Descriptors.MolWt(mol), 2),
        "LogP":           round(Descriptors.MolLogP(mol), 2),
        "HBD":            Lipinski.NumHDonors(mol),
        "HBA":            Lipinski.NumHAcceptors(mol),
        "TPSA":           round(Descriptors.TPSA(mol), 2),
        "RotB":           Lipinski.NumRotatableBonds(mol),
        "Rings":          Lipinski.RingCount(mol),
        "Aromatic_Rings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "Heavy_Atoms":    mol.GetNumHeavyAtoms(),
        "Fraction_Csp3":  round(rdMolDescriptors.CalcFractionCSP3(mol), 3),
        "Mol_Refract":    round(Descriptors.MolMR(mol), 2),
        "Formal_Charge":  Chem.GetFormalCharge(mol),
    }


# ─── Compute for each ligand ───
print("=" * 70)
print("  Molecular Descriptors")
print("=" * 70)

descriptors_data = {}

for lid, mol in ligands.items():
    props = compute_descriptors(mol)
    descriptors_data[lid] = props
    print(f"\n▶ {lid}:")
    for key, value in props.items():
        print(f"   {key:18s} = {value}")

# ─── Build DataFrame ───
df_descriptors = pd.DataFrame(descriptors_data).T
df_descriptors.index.name = "Ligand"

print("\n" + "=" * 70)
print("  Summary Table")
print("=" * 70)
print(df_descriptors.to_string())
print("=" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 5: Lipinski's Rule of 5
# ═══════════════════════════════════════════════════════════

# ─── Lipinski's thresholds ───
LIPINSKI_RULES = {
    "MW":   {"threshold": 500, "direction": "<=", "name": "Molecular Weight"},
    "LogP": {"threshold": 5,   "direction": "<=", "name": "Lipophilicity"},
    "HBD":  {"threshold": 5,   "direction": "<=", "name": "H-Bond Donors"},
    "HBA":  {"threshold": 10,  "direction": "<=", "name": "H-Bond Acceptors"},
}

def check_lipinski(props):
    """Check Lipinski's Rule of 5 for a molecule."""
    results = {}
    violations = 0
    
    for key, rule in LIPINSKI_RULES.items():
        value = props[key]
        threshold = rule["threshold"]
        passed = value <= threshold
        results[f"{key}_pass"] = passed
        if not passed:
            violations += 1
    
    results["violations"] = violations
    
    if violations == 0:
        results["verdict"] = "Excellent"
    elif violations == 1:
        results["verdict"] = "Acceptable"
    else:
        results["verdict"] = "Poor"
    
    return results


# ─── Apply to each ligand ───
print("=" * 75)
print("  Lipinski's Rule of 5 — Analysis")
print("=" * 75)

lipinski_data = {}

for lid, props in descriptors_data.items():
    lip = check_lipinski(props)
    lipinski_data[lid] = lip
    
    print(f"\n▶ {lid}:")
    for key, rule in LIPINSKI_RULES.items():
        value = props[key]
        threshold = rule["threshold"]
        icon = "✅" if lip[f"{key}_pass"] else "❌"
        print(f"   {icon} {rule['name']:20s} = {value:8} (limit: {threshold})")
    
    print(f"\n   Violations: {lip['violations']}")
    print(f"   Verdict:    {lip['verdict']}")

# ─── Summary Table ───
print("\n" + "=" * 75)
print("  Lipinski Summary Table")
print("=" * 75)

df_lipinski = pd.DataFrame(lipinski_data).T
df_lipinski = df_lipinski[["violations", "verdict"]]
print(df_lipinski.to_string())
print("=" * 75)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 6: Veber's Rules — Oral Bioavailability
# ═══════════════════════════════════════════════════════════

# ─── Veber's thresholds ───
VEBER_RULES = {
    "RotB": {"threshold": 10,  "name": "Rotatable Bonds"},
    "TPSA": {"threshold": 140, "name": "Topological Polar Surface Area"},
}

def check_veber(props):
    """Check Veber's Rules for oral bioavailability."""
    results = {}
    violations = 0
    
    for key, rule in VEBER_RULES.items():
        value = props[key]
        threshold = rule["threshold"]
        passed = value <= threshold
        results[f"{key}_pass"] = passed
        if not passed:
            violations += 1
    
    results["violations"] = violations
    results["verdict"] = "Pass" if violations == 0 else "Fail"
    
    return results


# ─── Apply to each ligand ───
print("=" * 75)
print("  Veber's Rules — Analysis")
print("=" * 75)

veber_data = {}

for lid, props in descriptors_data.items():
    veb = check_veber(props)
    veber_data[lid] = veb
    
    print(f"\n▶ {lid}:")
    for key, rule in VEBER_RULES.items():
        value = props[key]
        threshold = rule["threshold"]
        icon = "✅" if veb[f"{key}_pass"] else "❌"
        print(f"   {icon} {rule['name']:35s} = {value:8} (limit: {threshold})")
    
    print(f"\n   Violations: {veb['violations']}")
    print(f"   Verdict:    {veb['verdict']}")

# ─── Summary Table ───
print("\n" + "=" * 75)
print("  Veber Summary Table")
print("=" * 75)

df_veber = pd.DataFrame(veber_data).T
df_veber = df_veber[["violations", "verdict"]]
print(df_veber.to_string())
print("=" * 75)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 7: PAINS Filter — Detect Problematic Compounds
# ═══════════════════════════════════════════════════════════

# ─── Setup PAINS filter catalog ───
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog(params)

print("=" * 75)
print("  PAINS Filter Analysis")
print("=" * 75)
print("\nPAINS = Pan-Assay Interference Compounds")
print("These are compounds that give false positives in many assays.\n")

# ─── Check each ligand ───
pains_data = {}

for lid, mol in ligands.items():
    matches = pains_catalog.GetMatches(mol)
    n_alerts = len(matches)
    
    alerts_list = []
    for match in matches:
        alerts_list.append(match.GetDescription())
    
    pains_data[lid] = {
        "n_alerts": n_alerts,
        "alerts": alerts_list,
        "verdict": "Clean" if n_alerts == 0 else "Flagged",
    }
    
    print(f"\n▶ {lid}:")
    if n_alerts == 0:
        print(f"   ✅ Clean — No PAINS alerts")
    else:
        print(f"   ⚠️  {n_alerts} PAINS alert(s):")
        for alert in alerts_list[:5]:  # Show first 5
            print(f"      - {alert}")

# ─── Summary Table ───
print("\n" + "=" * 75)
print("  PAINS Summary Table")
print("=" * 75)

summary = {lid: data["n_alerts"] for lid, data in pains_data.items()}
df_pains = pd.DataFrame.from_dict(summary, orient="index", columns=["PAINS_Alerts"])
df_pains["Verdict"] = df_pains["PAINS_Alerts"].apply(
    lambda x: "✅ Clean" if x == 0 else "⚠️ Flagged"
)
print(df_pains.to_string())
print("=" * 75)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8: Radar Chart — Visual Comparison
# ═══════════════════════════════════════════════════════════

radar_props = [
    "MW",
    "LogP",
    "HBD",
    "HBA",
    "TPSA",
    "RotB",
]

max_vals = {
    "MW": 600,
    "LogP": 6,
    "HBD": 6,
    "HBA": 12,
    "TPSA": 200,
    "RotB": 15,
}

df_radar = df_descriptors[
    radar_props
].copy()

for prop in radar_props:
    df_radar[prop] = (
        df_radar[prop] / max_vals[prop]
    )

angles = np.linspace(
    0,
    2 * np.pi,
    len(radar_props),
    endpoint=False,
).tolist()

angles += angles[:1]

fig, ax = plt.subplots(
    figsize=(9, 9),
    subplot_kw=dict(polar=True),
)

for lid, row in df_radar.iterrows():
    values = row.tolist()
    values += values[:1]

    ax.plot(
        angles,
        values,
        "o-",
        linewidth=2,
        label=lid,
    )

    ax.fill(
        angles,
        values,
        alpha=0.12,
    )

ax.set_xticks(
    angles[:-1]
)

ax.set_xticklabels(
    radar_props,
    fontsize=12,
    fontweight="bold",
)

ax.set_ylim(0, 1)

ax.set_yticks(
    [0.25, 0.5, 0.75, 1.0]
)

ax.set_yticklabels(
    ["25%", "50%", "75%", "100%"],
    fontsize=9,
)

ax.set_title(
    "Rule-Based Molecular Properties — Current Top 5\n"
    "(Normalized to typical upper limits)",
    fontsize=14,
    fontweight="bold",
    pad=20,
)

ax.legend(
    loc="upper right",
    bbox_to_anchor=(1.3, 1.1),
    fontsize=10,
)

plt.tight_layout()

plt.savefig(
    "docking/results/admet_radar.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "✅ Saved: docking/results/admet_radar.png"
)


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 9: Bar Chart — Per-Property Comparison
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9),
)

axes = axes.flatten()

props_to_plot = [
    (
        "MW",
        "Molecular Weight (Da)",
        500,
        "Lipinski limit",
    ),
    (
        "LogP",
        "LogP (lipophilicity)",
        5,
        "Lipinski limit",
    ),
    (
        "HBD",
        "H-Bond Donors",
        5,
        "Lipinski limit",
    ),
    (
        "HBA",
        "H-Bond Acceptors",
        10,
        "Lipinski limit",
    ),
    (
        "TPSA",
        "TPSA (Å²)",
        140,
        "Veber limit",
    ),
    (
        "RotB",
        "Rotatable Bonds",
        10,
        "Veber limit",
    ),
]

ligand_ids = list(
    ligands.keys()
)

for i, (
    prop,
    ylabel,
    limit,
    limit_label,
) in enumerate(props_to_plot):

    ax = axes[i]

    values = [
        descriptors_data[lid][prop]
        for lid in ligand_ids
    ]

    bars = ax.bar(
        ligand_ids,
        values,
    )

    ax.axhline(
        y=limit,
        linestyle="--",
        linewidth=2,
        label=limit_label,
    )

    for bar, val in zip(
        bars,
        values,
    ):
        height = bar.get_height()

        offset = (
            max(values) * 0.03
            if max(values) > 0
            else 0.1
        )

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + offset,
            (
                f"{val:.2f}"
                if isinstance(val, float)
                else str(val)
            ),
            ha="center",
            fontsize=9,
            fontweight="bold",
        )

    ax.set_ylabel(
        ylabel,
        fontsize=11,
        fontweight="bold",
    )

    ax.set_title(
        prop,
        fontsize=13,
        fontweight="bold",
    )

    ax.legend(
        fontsize=9,
        loc="upper right",
    )

    ax.grid(
        axis="y",
        alpha=0.3,
    )

    ax.set_ylim(
        0,
        max(values + [limit]) * 1.25,
    )

plt.suptitle(
    "Rule-Based Molecular Properties — Current Top 5",
    fontsize=16,
    fontweight="bold",
    y=1.00,
)

plt.tight_layout()

plt.savefig(
    "docking/results/admet_barchart.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "✅ Saved: docking/results/admet_barchart.png"
)


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 10: Save Current Authoritative Results
# ═══════════════════════════════════════════════════════════

ADMET_DIR = "docking/results/admet"

os.makedirs(
    ADMET_DIR,
    exist_ok=True,
)

summary_data = []

for _, row in ranking.iterrows():

    lid = row["ligand_id"]

    lip = lipinski_data[lid]
    veb = veber_data[lid]
    pns = pains_data[lid]
    props = descriptors_data[lid]

    summary_data.append(
        {
            "rank": int(row.name + 1),
            "ligand_id": lid,
            "affinity": float(row["affinity"]),
            **props,
            "Lipinski_Violations": lip["violations"],
            "Lipinski_Verdict": lip["verdict"],
            "Veber_Violations": veb["violations"],
            "Veber_Verdict": veb["verdict"],
            "PAINS_Alerts": pns["n_alerts"],
            "PAINS_Verdict": pns["verdict"],
            "PAINS_Details": "; ".join(
                pns["alerts"]
            ),
            "source_pose": (
                f"{FIXED_DIR}/{lid}_fixed.sd"
            ),
        }
    )

df_final = pd.DataFrame(
    summary_data
)

df_final.to_csv(
    f"{ADMET_DIR}/admet_summary.csv",
    index=False,
)

df_final[
    [
        "ligand_id",
        "affinity",
        "MW",
        "LogP",
        "HBD",
        "HBA",
        "TPSA",
        "RotB",
        "Lipinski_Violations",
        "Lipinski_Verdict",
        "Veber_Violations",
        "Veber_Verdict",
        "PAINS_Alerts",
        "PAINS_Verdict",
    ]
].to_csv(
    f"{ADMET_DIR}/admet_top5.csv",
    index=False,
)

print(
    f"✅ Saved: "
    f"{ADMET_DIR}/admet_summary.csv"
)

print(
    f"✅ Saved: "
    f"{ADMET_DIR}/admet_top5.csv"
)

print(
    "ℹ️ Legacy component ADMET CSV files "
    "are not regenerated."
)


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 11: Final Summary and Scientific Interpretation
# ═══════════════════════════════════════════════════════════

print("=" * 100)

print(
    "  FINAL RULE-BASED PROPERTY SUMMARY — CURRENT TOP 5"
)

print("=" * 100)

print(
    "\nComplete Property Table:"
)

print("-" * 100)

print(
    df_final.to_string(index=False)
)

print("-" * 100)

print(
    "\nRule-Based Assessment per Ligand:"
)

print("=" * 100)

for lid in TOP_HITS:

    lip = lipinski_data[lid]
    veb = veber_data[lid]
    pns = pains_data[lid]
    props = descriptors_data[lid]

    favorable = (
        lip["violations"] <= 1
        and veb["verdict"] == "Pass"
        and pns["n_alerts"] == 0
    )

    verdict = (
        "Favorable"
        if favorable
        else "Needs Review"
    )

    print(
        f"\n▶ {lid} — {verdict}"
    )

    print(
        f"   MW:     {props['MW']:.2f} Da"
    )

    print(
        f"   LogP:   {props['LogP']:.2f}"
    )

    print(
        f"   HBD:    {props['HBD']}"
    )

    print(
        f"   HBA:    {props['HBA']}"
    )

    print(
        f"   TPSA:   {props['TPSA']:.2f} Å²"
    )

    print(
        f"   RotB:   {props['RotB']}"
    )

    print(
        f"   Lipinski: "
        f"{lip['violations']} violations "
        f"→ {lip['verdict']}"
    )

    print(
        f"   Veber: "
        f"{veb['verdict']}"
    )

    print(
        f"   PAINS: "
        f"{pns['n_alerts']} alerts "
        f"→ {pns['verdict']}"
    )

print(
    "\n" + "=" * 100
)

print(
    "Scientific scope:"
)

print(
    "These are RDKit descriptors and "
    "rule-based filters, not full ADME "
    "or toxicity predictions."
)

print(
    "A favorable rule-based profile does "
    "not establish experimental efficacy, "
    "safety, or oral exposure."
)

print("=" * 100)


---

## 🎉 Lesson 15 — Summary

### ✅ What We Learned

**1. Molecular Descriptors (RDKit)**
- Calculated molecular properties for the current standardized Top 5.
- Examples include MW, LogP, HBD, HBA, TPSA, RotB, ring counts, and formal charge.

**2. Rule-Based Drug-Likeness Filters**
- **Lipinski's Rule of Five**
- **Veber rules**
- **PAINS filtering**

**3. Current Results**
- The Top 5 are derived dynamically from `standardized_affinities.csv`.
- The current analysis covers **R11, 13U, BAH, 12U, and 607**.
- Current authoritative summaries are saved to:
  - `docking/results/admet/admet_summary.csv`
  - `docking/results/admet/admet_top5.csv`

### Scientific Interpretation

These analyses are complementary to docking and ProLIF results.

They are **rule-based molecular-property assessments**, not full ADME or toxicity predictions, and they do not provide experimental validation of binding, efficacy, safety, or oral exposure.

---

*End of Lesson 15* 🎓
